# Initialization

In [2]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [3]:
%matplotlib inline
%config InlineBackend.figure_format = 'png'
%config InlineBackend.figure_format = 'retina'

# Загрузка данных

In [4]:
items = pd.read_parquet("items.parquet")
events = pd.read_parquet("events.parquet")

# Разбиение с учётом хронологии

Рекомендательные системы на практике работают с учётом хронологии. Поэтому поток событий для тренировки и валидации полезно делить на то, что уже случилось, и что ещё случится. Это позволяет проводить валидацию на тех же пользователях, на которых тренировались, но на их событиях в будущем.

# === Знакомство: "холодный" старт

In [5]:
# зададим точку разбиения
train_test_global_time_split_date = pd.to_datetime("2017-08-01").date()

train_test_global_time_split_idx = (
    pd.to_datetime(events["started_at"]).dt.date
    < train_test_global_time_split_date
)

# разбиение
events_train = events[train_test_global_time_split_idx].copy()
events_test = events[~train_test_global_time_split_idx].copy()

# пользователи в train и test
users_train = events_train["user_id"].drop_duplicates()
users_test = events_test["user_id"].drop_duplicates()

# пользователи, которые есть и в train, и в test
common_users = set(users_train).intersection(set(users_test))

print(len(users_train), len(users_test), len(common_users))

428220 123223 120858


In [6]:
# Сколько уникальных пользователей есть и в events_train, и в events_test?
print(events_train.user_id.nunique())
print(events_test.user_id.nunique())

428220
123223


In [7]:
common_users = set(events_train["user_id"]).intersection(
    set(events_test["user_id"])
)

len(common_users)

120858

In [8]:
# Идентифицируйте холодных пользователей и оцените их количество.
# Холодные пользователи (cold users) — это пользователи, которые есть в events_test, но отсутствуют в events_train.
cold_users = set(events_test["user_id"]) - set(events_train["user_id"])

print(len(cold_users)) 

2365


# === Знакомство: первые персональные рекомендации

In [8]:
# получить топ-100 наиболее популярных книг
from sklearn.preprocessing import MinMaxScaler

top_pop_start_date = pd.to_datetime("2015-01-01").date()

item_popularity = events_train \
    .query("started_at >= @top_pop_start_date") \
    .groupby(["item_id"]).agg(users=("user_id", "nunique"), avg_rating=("rating", "mean")).reset_index()

# нормализация пользователей и среднего рейтинга, требуется для их приведения к одному масштабу
scaler = MinMaxScaler()
item_popularity[["users_norm", "avg_rating_norm"]] = scaler.fit_transform(
    item_popularity[["users", "avg_rating"]]
)

# вычисляем popularity_score, как скор популярности со штрафом за низкий рейтинг
item_popularity["popularity_score"] = (
    item_popularity["users_norm"] * item_popularity["avg_rating_norm"]
)

# сортируем по убыванию popularity_score
item_popularity = item_popularity.sort_values(
    by="popularity_score",
    ascending=False
)

# выбираем первые 100 айтемов со средней оценкой avg_rating не меньше 4
top_k_pop_items = (
    item_popularity
    .query("avg_rating >= 4")
    .head(100)
)

In [9]:
# Сколько пользователей оценило книгу, попавшую на первое место в top_k_pop_items?
top_item_id = top_k_pop_items.iloc[0]["item_id"]

item_popularity.loc[
    item_popularity["item_id"] == top_item_id,
    "users"
].iloc[0]

20207

In [10]:
# добавляем информацию о книгах
top_k_pop_items = top_k_pop_items.merge(
    items.set_index("item_id")[["author", "title", "genre_and_votes", "publication_year"]], on="item_id")

with pd.option_context('display.max_rows', 100):
    display(top_k_pop_items[["item_id", "author", "title", "publication_year", "users", "avg_rating", "popularity_score", "genre_and_votes"]])

,item_id,author,title,publication_year,users,avg_rating,popularity_score,genre_and_votes
0,18007564,Andy Weir,The Martian,2014,20207,4.321275,0.412333,"{'Science Fiction': 11966, 'Fiction': 8430}"
1,18143977,Anthony Doerr,All the Light We Cannot See,2014,19462,4.290669,0.393471,"{'Historical-Historical Fiction': 13679, 'Fict..."
2,3,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,1997,15139,4.706057,0.344702,"{'Fantasy': 59818, 'Fiction': 17918, 'Young Ad..."
3,16096824,Sarah J. Maas,A Court of Thorns and Roses (A Court of Thorns...,2015,16770,4.301014,0.340108,"{'Fantasy': 14326, 'Young Adult': 4662, 'Roman..."
4,15881,"J.K. Rowling, Mary GrandPré",Harry Potter and the Chamber of Secrets (Harry...,1999,13043,4.632447,0.291076,"{'Fantasy': 50130, 'Young Adult': 15202, 'Fict..."
5,38447,Margaret Atwood,The Handmaid's Tale,1998,14611,4.232770,0.290194,"{'Fiction': 15424, 'Classics': 9937, 'Science ..."
6,11235712,Marissa Meyer,"Cinder (The Lunar Chronicles, #1)",2012,14348,4.179189,0.280247,"{'Young Adult': 10539, 'Fantasy': 9237, 'Scien..."
7,17927395,Sarah J. Maas,A Court of Mist and Fury (A Court of Thorns an...,2016,12177,4.730640,0.279094,"{'Fantasy': 10186, 'Romance': 3346, 'Young Adu..."
8,5,"J.K. Rowling, Mary GrandPré",Harry Potter and the Prisoner of Azkaban (Harr...,2004,11890,4.770143,0.275401,"{'Fantasy': 49784, 'Young Adult': 15393, 'Fict..."
9,13206900,Marissa Meyer,"Winter (The Lunar Chronicles, #4)",2015,12291,4.534293,0.266881,"{'Fantasy': 4835, 'Young Adult': 4672, 'Scienc..."


Для какой доли событий «холодных» пользователей 

в events_test рекомендации 

в top_k_pop_items совпали по книгам? 

Округлите ответ до сотых.

In [11]:
cold_users_events_with_recs = \
    events_test[events_test["user_id"].isin(cold_users)] \
    .merge(top_k_pop_items[["item_id", "avg_rating"]], on="item_id", how="left")

cold_user_items_no_avg_rating_idx = cold_users_events_with_recs["avg_rating"].isnull()

cold_user_recs = cold_users_events_with_recs[~cold_user_items_no_avg_rating_idx] \
    [["user_id", "item_id", "rating", "avg_rating"]]

In [12]:
share = (
    events_test[events_test["user_id"].isin(cold_users)]
    .merge(top_k_pop_items[["item_id", "avg_rating"]], on="item_id", how="left")["avg_rating"]
    .notna()
    .mean()
)

round(share, 2)

0.2

Посчитайте метрики rmse и mae для полученных рекомендаций.

In [13]:
# посчитаем метрики рекомендаций
from sklearn.metrics import mean_squared_error, mean_absolute_error

rmse = mean_squared_error(
    cold_user_recs["rating"],
    cold_user_recs["avg_rating"],
    squared=False
)

mae = mean_absolute_error(
    cold_user_recs["rating"],
    cold_user_recs["avg_rating"]
)

print(round(rmse, 2), round(mae, 2))

0.78 0.62


In [14]:
# посчитаем покрытие холодных пользователей рекомендациями

cold_users_hit_ratio = cold_users_events_with_recs.groupby("user_id").agg(hits=("avg_rating", lambda x: (~x.isnull()).mean()))

print(f"Доля пользователей без релевантных рекомендаций: {(cold_users_hit_ratio == 0).mean().iat[0]:.2f}")
print(f"Среднее покрытие пользователей: {cold_users_hit_ratio[cold_users_hit_ratio != 0].mean().iat[0]:.2f}")

Доля пользователей без релевантных рекомендаций: 0.59
Среднее покрытие пользователей: 0.44


# === Базовые подходы: коллаборативная фильтрация

In [15]:
# Матрица пользователей и объектов U - I
# Оценка степени разреженности матрици U - I
n_users = events["user_id"].nunique()
n_items = events["item_id"].nunique()
n_interactions = len(events)

sparsity = 1 - n_interactions / (n_users * n_items)
round(sparsity * 100, 1)

99.9

Вывод:

матрица сильно разрежена

## Реализация SVD-алгоритма

In [16]:
from surprise import Dataset, Reader
from surprise import SVD

# используем Reader из библиотеки surprise для преобразования событий (events)
# в формат, необходимый surprise
reader = Reader(rating_scale=(1, 5))
surprise_train_set = Dataset.load_from_df(events_train[['user_id', 'item_id', 'rating']], reader)
surprise_train_set = surprise_train_set.build_full_trainset()

# инициализируем модель
svd_model = SVD(n_factors=100, random_state=0)

# обучаем модель
svd_model.fit(surprise_train_set)

In [17]:
surprise_test_set = list(events_test[['user_id', 'item_id', 'rating']].itertuples(index=False))

# получаем рекомендации для тестовой выборки
svd_predictions = svd_model.test(surprise_test_set)

### Оценка рекомендаций

In [18]:
from surprise import accuracy

rmse = accuracy.rmse(svd_predictions)
mae = accuracy.mae(svd_predictions)

print(rmse, mae)

RMSE: 0.8289
MAE:  0.6474
0.8288711689059135 0.647437483750257


### Проверка метрик на адекватность

In [19]:
from surprise import NormalPredictor

# инициализируем состояние генератора, это необходимо для получения
# одной и той же последовательности случайных чисел, только в учебных целях
np.random.seed(0)

random_model = NormalPredictor()

random_model.fit(surprise_train_set)
random_predictions = random_model.test(surprise_test_set)

In [20]:
from surprise import accuracy

rmse_r = accuracy.rmse(random_predictions)
mae_r = accuracy.mae(random_predictions)

print(rmse_r, mae_r)

RMSE: 1.2628
MAE:  1.0018
1.2628030301013033 1.0017726877569562


На сколько процентов MAE для случайных рекомендаций от NormalPredictor выше значения MAE от SVD?

In [21]:
from sklearn.metrics import mean_absolute_error

# преобразуем predictions в список реальных и предсказанных рейтингов
y_true = [pred.r_ui for pred in random_predictions]
y_pred = [pred.est for pred in random_predictions]

mae_random = mean_absolute_error(y_true, y_pred)
round(mae_random, 4)

1.0018

In [22]:
# Посчитаем разницу в метриках
percent_diff = (mae_random - mae) / mae * 100
round(percent_diff)

55

Удалите из events события для редких айтемов — таких, с которыми взаимодействовало менее N пользователей. Возьмите небольшое N, например 2-3 пользователя. Получите рекомендации, посчитайте метрики, оцените, как они изменились. Подумайте, с чем могут быть связаны такие изменения.

### Удаляем редкие айтемы

Возьмём, например, N = 3 (айтем должен иметь ≥ 3 пользователей)

In [23]:
N = 3

# считаем число уникальных пользователей на каждый item
item_user_counts = events.groupby("item_id")["user_id"].nunique()

# оставляем только популярные айтемы
popular_items = item_user_counts[item_user_counts >= N].index

events_filtered = events[events["item_id"].isin(popular_items)]

### Проверим, как изменилась разреженность

In [24]:
# Проверим, как изменилась разреженность
n_users = events_filtered["user_id"].nunique()
n_items = events_filtered["item_id"].nunique()
n_interactions = len(events_filtered)

sparsity = 1 - n_interactions / (n_users * n_items)
round(sparsity * 100, 2)

99.93

### Обучаем модель заново

Обычно разреженность снижается, потому что мы убрали “хвост” из редких объектов.

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import numpy as np

reader = Reader(rating_scale=(events_filtered.rating.min(),
                              events_filtered.rating.max()))

data = Dataset.load_from_df(
    events_filtered[["user_id", "item_id", "rating"]],
    reader
)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

model = SVD(random_state=42)
model.fit(trainset)

predictions = model.test(testset)

y_true = [p.r_ui for p in predictions]
y_pred = [p.est for p in predictions]

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(np.mean((np.array(y_true) - np.array(y_pred))**2))

round(rmse, 3), round(mae, 3)

Выводы:

метрики улучшились.

- модель стала ошибаться меньше

- предсказания стали стабильнее

- шум уменьшился

Почему так произошло:

Убрали long-tail

Редкие айтемы имеют 1–2 взаимодействия → их параметры плохо оцениваются → модель часто ошибается.

То есть произошло улучшение accuracy, но возможное ухудшение diversity и coverage.

### Создайте функцию, которая позволит получить рекомендации для конкретного пользователя, 

### используя описанный подход. Дополните прекод.

In [11]:
import pandas as pd

def get_recommendations_svd(user_id, all_items, events, model, include_seen=True, n=5):
    """Возвращает n рекомендаций для user_id на основе surprise SVD."""

    # каталог всех книг: можно передать DataFrame/Series/список/множество
    if isinstance(all_items, pd.DataFrame):
        all_items_set = set(all_items["item_id"].unique())
    elif isinstance(all_items, pd.Series):
        all_items_set = set(all_items.unique())
    else:
        all_items_set = set(all_items)

    # какие айтемы предсказываем
    if include_seen:
        items_to_predict = list(all_items_set)
    else:
        seen_items = set(events[events["user_id"] == user_id]["item_id"].unique())
        items_to_predict = list(all_items_set - seen_items)

    # предсказания рейтингов для всех кандидатов
    predictions = [model.predict(user_id, item_id) for item_id in items_to_predict]

    # топ-n по убыванию score (pred.est)
    recommendations = sorted(predictions, key=lambda x: x.est, reverse=True)[:n]

    return pd.DataFrame(
        [(pred.iid, pred.est) for pred in recommendations],
        columns=["item_id", "score"]
    )

In [12]:
get_recommendations_svd(1296647, items, events_test, svd_model)

NameError: name 'svd_model' is not defined

In [13]:
# Третья рекомендация по убыванию score (строка с индексом 2) имеет:

item_id = 30688013

In [14]:
# выберем произвольного пользователя из тренировочной выборки ("прошлого")
user_id = events_train['user_id'].sample().iat[0]

print(f"user_id: {user_id}")

print("История (последние события, recent)")
user_history = (
    events_train
    .query("user_id == @user_id")
    .merge(items.set_index("item_id")[["author", "title", "genre_and_votes"]], on="item_id")
)
user_history_to_print = user_history[["author", "title", "started_at", "read_at", "rating", "genre_and_votes"]].tail(10)
display(user_history_to_print)

print("Рекомендации")
user_recommendations = get_recommendations_svd(user_id, items, events_train, svd_model)
user_recommendations = user_recommendations.merge(items[["item_id", "author", "title", "genre_and_votes"]], on="item_id")
display(user_recommendations)

user_id: 1270091
История (последние события, recent)


,author,title,started_at,read_at,rating,genre_and_votes
229,Maggie Stiefvater,"Linger (The Wolves of Mercy Falls, #2)",2010-09-27,2010-10-01,5,"{'Young Adult': 3135, 'Fantasy': 2399, 'Romanc..."
230,Richelle Mead,"Spirit Bound (Vampire Academy, #5)",2010-06-06,2010-06-13,5,"{'Young Adult': 3651, 'Paranormal-Vampires': 3..."
231,Charlaine Harris,"Dead in the Family (Sookie Stackhouse, #10)",2010-05-18,2010-05-22,3,"{'Fantasy': 2125, 'Paranormal-Vampires': 1966,..."
232,J.R. Ward,"Lover Mine (Black Dagger Brotherhood, #8)",2010-08-16,2010-08-25,5,"{'Fantasy-Paranormal': 1588, 'Romance-Paranorm..."
233,Patricia Briggs,"Moon Called (Mercy Thompson, #1)",2010-11-24,2010-11-28,4,"{'Fantasy-Urban Fantasy': 4805, 'Fantasy': 355..."
234,Richelle Mead,"Succubus Blues (Georgina Kincaid, #1)",2010-06-12,2010-06-21,5,"{'Fantasy-Urban Fantasy': 1419, 'Fantasy-Paran..."
235,J.R. Ward,"Covet (Fallen Angels, #1)",2010-01-18,2010-01-19,4,"{'Fantasy-Paranormal': 1002, 'Romance-Paranorm..."
236,Gena Showalter,The Darkest Whisper (Lords of the Underworld #4),2011-09-05,2011-09-09,5,"{'Romance-Paranormal Romance': 950, 'Fantasy-P..."
237,Patricia Briggs,"Hunting Ground (Alpha & Omega, #2)",2011-09-05,2011-09-05,5,"{'Fantasy-Urban Fantasy': 2118, 'Fantasy': 120..."
238,Melissa Marr,"Wicked Lovely (Wicked Lovely, #1)",2010-04-24,2010-04-26,4,"{'Fantasy': 3440, 'Young Adult': 3189, 'Romanc..."


Рекомендации


NameError: name 'svd_model' is not defined

Ниже приведён код для перекодировки идентификаторов. Дополните его для перекодировки идентификаторов объектов, а затем выполните.

In [15]:
import scipy
import sklearn.preprocessing

# перекодируем идентификаторы пользователей
user_encoder = sklearn.preprocessing.LabelEncoder()
user_encoder.fit(events["user_id"])

events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])

# перекодируем идентификаторы объектов
item_encoder = sklearn.preprocessing.LabelEncoder()
item_encoder.fit(items["item_id"])   # fit на полном каталоге items

items["item_id_enc"] = item_encoder.transform(items["item_id"])

events_train["item_id_enc"] = item_encoder.transform(events_train["item_id"])
events_test["item_id_enc"] = item_encoder.transform(events_test["item_id"])

In [16]:
# Правальный ответ 43304
events_train['item_id_enc'].max()

43304

In [17]:
# создаём sparse-матрицу формата CSR 
user_item_matrix_train = scipy.sparse.csr_matrix((
    events_train["rating"],
    (events_train['user_id_enc'], events_train['item_id_enc'])),
    dtype=np.int8)

In [18]:
import sys

sum([sys.getsizeof(i) for i in user_item_matrix_train.data])/1024**3

0.26370687410235405

In [19]:
from implicit.als import AlternatingLeastSquares

als_model = AlternatingLeastSquares(factors=50, iterations=50, regularization=0.05, random_state=0)
als_model.fit(user_item_matrix_train)

/home/mle_projects/mle-recsys-start/env_recsys_start/lib/python3.10/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/50 [00:00<?, ?it/s]

In [20]:
def get_recommendations_als(user_item_matrix, model, user_id, user_encoder, item_encoder, include_seen=True, n=5):
    """
    Возвращает отранжированные рекомендации для заданного пользователя
    """
    user_id_enc = user_encoder.transform([user_id])[0]
    recommendations = model.recommend(
         user_id_enc, 
         user_item_matrix[user_id_enc], 
         filter_already_liked_items=not include_seen,
         N=n)
    recommendations = pd.DataFrame({"item_id_enc": recommendations[0], "score": recommendations[1]})
    recommendations["item_id"] = item_encoder.inverse_transform(recommendations["item_id_enc"])

    return recommendations

In [21]:
# получаем список всех возможных user_id (перекодированных)
user_ids_encoded = range(len(user_encoder.classes_))

# получаем рекомендации для всех пользователей
als_recommendations = als_model.recommend(
    user_ids_encoded, 
    user_item_matrix_train[user_ids_encoded], 
    filter_already_liked_items=False, N=100)

In [22]:
# преобразуем полученные рекомендации в табличный формат
item_ids_enc = als_recommendations[0]
als_scores = als_recommendations[1]

als_recommendations = pd.DataFrame({
    "user_id_enc": user_ids_encoded,
    "item_id_enc": item_ids_enc.tolist(), 
    "score": als_scores.tolist()})
als_recommendations = als_recommendations.explode(["item_id_enc", "score"], ignore_index=True)

# приводим типы данных
als_recommendations["item_id_enc"] = als_recommendations["item_id_enc"].astype("int")
als_recommendations["score"] = als_recommendations["score"].astype("float")

# получаем изначальные идентификаторы
als_recommendations["user_id"] = user_encoder.inverse_transform(als_recommendations["user_id_enc"])
als_recommendations["item_id"] = item_encoder.inverse_transform(als_recommendations["item_id_enc"])
als_recommendations = als_recommendations.drop(columns=["user_id_enc", "item_id_enc"])

In [23]:
als_recommendations = als_recommendations[["user_id", "item_id", "score"]]
als_recommendations.to_parquet("als_recommendations.parquet")

In [24]:
als_recommendations = (
    als_recommendations
    .merge(events_test[["user_id", "item_id", "rating"]]
               .rename(columns={"rating": "rating_test"}), 
           on=["user_id", "item_id"], how="left")
)

In [25]:
import sklearn.metrics

def compute_ndcg(rating: pd.Series, score: pd.Series, k):

    """ подсчёт ndcg
    rating: истинные оценки
    score: оценки модели
    k: количество айтемов (по убыванию score) для оценки, остальные - отбрасываются
    """

    # если кол-во объектов меньше 2, то NDCG - не определена
    if len(rating) < 2:
        return np.nan

    ndcg = sklearn.metrics.ndcg_score(np.asarray([rating.to_numpy()]), np.asarray([score.to_numpy()]), k=k)

    return ndcg

In [26]:
rating_test_idx = ~als_recommendations["rating_test"].isnull()
ndcg_at_5_scores = als_recommendations[rating_test_idx].groupby("user_id").apply(lambda x: compute_ndcg(x["rating_test"], x["score"], k=5))

In [27]:
print(round(ndcg_at_5_scores.mean(), 2))

0.98


# === Базовые подходы: контентные рекомендации

In [42]:
items["genre_and_votes"] = items["genre_and_votes"].apply(eval)

In [43]:
def get_genres(items):

    """ 
    извлекает список жанров по всем книгам,
    подсчитывает долю голосов по каждому из них
    """

    genres_counter = {}

    for k, v in items.iterrows():
        genre_and_votes = v["genre_and_votes"]

        if genre_and_votes is None or not isinstance(genre_and_votes, dict):
            continue

        for genre, votes in genre_and_votes.items():
            try:
                genres_counter[genre] += votes
            except KeyError:
                genres_counter[genre] = votes

    genres = pd.Series(genres_counter, name="votes")
    genres = genres.to_frame()
    genres = genres.reset_index().rename(columns={"index": "name"})
    genres.index.name = "genre_id"

    return genres


genres = get_genres(items)

In [44]:
genres["score"] = genres["votes"] / genres["votes"].sum()
genres.sort_values(by="score", ascending=False).head(10)

,name,votes,score
genre_id,,,
25,Fantasy,6850115,0.149498
1,Fiction,6406698,0.139821
38,Classics,3415071,0.074531
18,Young Adult,3297027,0.071955
34,Romance,2422690,0.052873
5,Nonfiction,1737798,0.037926
16,Historical-Historical Fiction,1531489,0.033423
20,Mystery,1371370,0.029929
24,Science Fiction,1218997,0.026604


In [45]:
def get_item2genre_matrix(genres, items):

    genre_names_to_id = genres.reset_index().set_index("name")["genre_id"].to_dict()

    # list to build CSR matrix
    genres_csr_data = []
    genres_csr_row_idx = []
    genres_csr_col_idx = []

    for item_idx, (k, v) in enumerate(items.iterrows()):
        if v["genre_and_votes"] is None:
            continue
        for genre_name, votes in v["genre_and_votes"].items():
            genre_idx = genre_names_to_id[genre_name]
            genres_csr_data.append(int(votes))
            genres_csr_row_idx.append(item_idx)
            genres_csr_col_idx.append(genre_idx)

    genres_csr = scipy.sparse.csr_matrix((genres_csr_data, (genres_csr_row_idx, genres_csr_col_idx)), shape=(len(items), len(genres)))
    # нормализуем, чтобы сумма оценок принадлежности к жанру была равна 1
    genres_csr = sklearn.preprocessing.normalize(genres_csr, norm='l1', axis=1)

    return genres_csr

In [46]:
genre_matrix = get_item2genre_matrix(genres, items)
genre_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 210895 stored elements and shape (43312, 815)>

In [47]:
items = items.sort_values(by="item_id_enc")
all_items_genres_csr = get_item2genre_matrix(genres, items)

In [48]:
user_id = 1000010
user_events = events_train.query("user_id == @user_id")[["item_id", "rating"]]
user_items = items[items["item_id"].isin(user_events["item_id"])]

user_items_genres_csr = get_item2genre_matrix(genres, user_items)
user_items_genres_csr

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 149 stored elements and shape (22, 815)>

In [49]:
# вычислим склонность пользователя к жанрам как среднее взвешенное значение популяции на его оценки книг.

# преобразуем пользовательские оценки из списка в вектор-столбец
user_ratings = user_events["rating"].to_numpy() / 5
user_ratings = np.expand_dims(user_ratings, axis=1)

user_items_genres_weighted = user_items_genres_csr.multiply(user_ratings)

user_genres_scores = np.asarray(user_items_genres_weighted.mean(axis=0))

In [50]:
# выведем список жанров, которые предпочитает пользователь

user_genres = genres.copy()
user_genres["score"] = np.ravel(user_genres_scores)
user_genres = user_genres[user_genres["score"] > 0].sort_values(by=["score"], ascending=False)

user_genres.head(5)

,name,votes,score
genre_id,,,
1,Fiction,6406698,0.185241
38,Classics,3415071,0.103879
25,Fantasy,6850115,0.072447
5,Nonfiction,1737798,0.050865
24,Science Fiction,1218997,0.040920


In [51]:
from sklearn.metrics.pairwise import cosine_similarity

# вычисляем сходство между вектором пользователя и векторами по книгам
similarity_scores = cosine_similarity(all_items_genres_csr, user_genres_scores)

# преобразуем в одномерный массив
similarity_scores = similarity_scores.flatten()

# получаем индексы top-k (по убыванию значений), по сути, индексы книг (encoded)
k = 5
top_k_indices = np.argsort(similarity_scores)[::-1][:k]

In [52]:
top_k = 10

top_k_indices = np.argsort(similarity_scores)[::-1][:top_k]

selected_items = items[items["item_id_enc"].isin(top_k_indices)]

with pd.option_context("max_colwidth", 100):
    display(selected_items[["author", "title", "genre_and_votes"]])

,author,title,genre_and_votes
163077,Aldous Huxley,Island,"{'Fiction': 885, 'Classics': 363, 'Science Fiction': 269, 'Philosophy': 196, 'Science Fiction-Dy..."
564712,Ray Bradbury,"Farewell Summer (Green Town, #3)","{'Fiction': 170, 'Fantasy': 72, 'Science Fiction': 72, 'Classics': 52}"
1358935,John Fowles,The Magus,"{'Fiction': 1204, 'Classics': 421, 'Fantasy': 228, 'Mystery': 203, 'Literature': 167}"
80465,G.K. Chesterton,The Napoleon of Notting Hill,"{'Fiction': 166, 'Classics': 88, 'Fantasy': 44, 'Humor': 22, 'Literature': 20}"
1168335,Ray Bradbury,"Dandelion Wine (Green Town, #1)","{'Fiction': 1438, 'Classics': 914, 'Science Fiction': 529, 'Fantasy': 456, 'Young Adult': 212}"
1629410,"Richard Bach, Russell Munson",Jonathan Livingston Seagull,"{'Fiction': 2540, 'Classics': 1262, 'Philosophy': 783, 'Fantasy': 443, 'Spirituality': 366, 'Ins..."
393210,"G.K. Chesterton, Jonathan Lethem",The Man Who Was Thursday: A Nightmare,"{'Fiction': 1257, 'Classics': 929, 'Mystery': 469, 'Fantasy': 293, 'Philosophy': 156, 'Literatur..."
2244467,Samuel Butler,"Erewhon (Erewhon , #1)","{'Fiction': 162, 'Classics': 139, 'Science Fiction': 60, 'Fantasy': 55}"
1386929,"Jack London, Lorenzo Carcaterra",The Star Rover,"{'Fiction': 109, 'Classics': 100, 'Science Fiction': 31, 'Fantasy': 29, 'Literature-American': 1..."
39408,"Paulo Coelho, Alan R. Clarke, James Noel Smith",The Alchemist,"{'Fiction': 14023, 'Classics': 5787, 'Fantasy': 3289, 'Philosophy': 2759}"


# === Базовые подходы: валидация

In [28]:
def process_events_recs_for_binary_metrics(events_train, events_test, recs, top_k=None):

    """
    размечает пары <user_id, item_id> для общего множества пользователей признаками
    - gt (ground truth)
    - pr (prediction)
    top_k: расчёт ведётся только для top k-рекомендаций
    """

    events_test["gt"] = True
    common_users = set(events_test["user_id"]) & set(recs["user_id"])

    print(f"Common users: {len(common_users)}")

    events_for_common_users = events_test[events_test["user_id"].isin(common_users)].copy()
    recs_for_common_users = recs[recs["user_id"].isin(common_users)].copy()

    recs_for_common_users = recs_for_common_users.sort_values(["user_id", "score"], ascending=[True, False])

    # оставляет только те item_id, которые были в events_train, 
    # т. к. модель не имела никакой возможности давать рекомендации для новых айтемов
    events_for_common_users = events_for_common_users[events_for_common_users["item_id"].isin(events_train["item_id"].unique())]

    if top_k is not None:
        recs_for_common_users = recs_for_common_users.groupby("user_id").head(top_k)

    events_recs_common = events_for_common_users[["user_id", "item_id", "gt"]].merge(
        recs_for_common_users[["user_id", "item_id", "score"]], 
        on=["user_id", "item_id"], how="outer")    

    events_recs_common["gt"] = events_recs_common["gt"].fillna(False)
    events_recs_common["pr"] = ~events_recs_common["score"].isnull()

    events_recs_common["tp"] = events_recs_common["gt"] & events_recs_common["pr"]
    events_recs_common["fp"] = ~events_recs_common["gt"] & events_recs_common["pr"]
    events_recs_common["fn"] = events_recs_common["gt"] & ~events_recs_common["pr"]

    return events_recs_common

In [29]:
events_recs_for_binary_metrics = process_events_recs_for_binary_metrics(
  events_train,
    events_test, 
    als_recommendations, 
    top_k=5)

Common users: 123223


In [30]:
def compute_cls_metrics(events_recs_for_binary_metric):

    groupper = events_recs_for_binary_metric.groupby("user_id")

    # precision = tp / (tp + fp)
    precision = groupper["tp"].sum() / (groupper["tp"].sum() + groupper["fp"].sum())
    precision = precision.fillna(0).mean()

    # recall = tp / (tp + fn)
    recall = groupper["tp"].sum() / (groupper["tp"].sum() + groupper["fn"].sum())
    recall = recall.fillna(0).mean()

    return precision, recall

In [31]:
compute_cls_metrics(events_recs_for_binary_metrics)

(0.007581376853347184, 0.014121568795222568)

# === Двухстадийный подход: метрики

In [61]:
# расчёт покрытия по объектам
events_recs_for_binary_metric = process_events_recs_for_binary_metrics(
    events_train, events_test, als_recommendations
)

cov_items = (
    events_recs_for_binary_metric.loc[events_recs_for_binary_metric["pr"], "item_id"].nunique()
    / items["item_id"].nunique()
)

print(f"{cov_items:.2f}")

Common users: 123223
0.09


In [ ]:
# расчёт покрытия по объектам
cov_items = als_recommendations["item_id"].nunique() / items["item_id"].nunique()
print(f"{cov_items:.2f}")

0.09


In [64]:
# Новизна (англ. Novelty)
# разметим каждую рекомендацию признаком read
events_train["read"] = True
als_recommendations = als_recommendations.merge(
    events_train[["user_id", "item_id", "read"]],
    on=["user_id", "item_id"],
    how="left"
)
als_recommendations["read"] = als_recommendations["read"].fillna(False).astype("bool")

# проставим ранги
als_recommendations = als_recommendations.sort_values(["user_id", "score"], ascending=[True, False])
als_recommendations["rank"] = als_recommendations.groupby("user_id").cumcount() + 1

# посчитаем novelty по пользователям
novelty_5 = (1 - als_recommendations.query("rank <= 5").groupby("user_id")["read"].mean())

# посчитаем средний novelty
print(round(novelty_5.mean(), 2))

0.61


In [ ]:
# Разнообразие (англ. Diversity)


# === Двухстадийный подход: модель

In [35]:
# задаём точку разбиения
split_date_for_labels = pd.to_datetime("2017-09-15").date()

split_date_for_labels_idx = events_test["started_at"] < split_date_for_labels

events_labels = events_test[split_date_for_labels_idx].copy()
events_test_2 = events_test[~split_date_for_labels_idx].copy()

In [36]:
events_labels['user_id'].nunique()

99849

In [37]:
# загружаем рекомендации от двух базовых генераторов
als_recommendations = pd.read_parquet("goodsread/candidates/training/als_recommendations.parquet")
content_recommendations = pd.read_parquet("goodsread/candidates/training/content_recommendations.parquet")

candidates = pd.merge(
    als_recommendations[["user_id", "item_id", "score"]].rename(columns={"score": "als_score"}),
    content_recommendations[["user_id", "item_id", "score"]].rename(columns={"score": "cnt_score"}),
    on=["user_id", "item_id"],
    how="outer"
)

In [38]:
candidates.shape

(82993094, 4)

In [39]:
# добавляем таргет к кандидатам со значением:
# — 1 для тех item_id, которые пользователь прочитал
# — 0, для всех остальных 

events_labels["target"] = 1
candidates = candidates.merge(
    events_labels[["user_id", "item_id", "target"]],
    on=["user_id", "item_id"],
    how="left"
)
candidates["target"] = candidates["target"].fillna(0).astype("int")

# в кандидатах оставляем только тех пользователей, у которых есть хотя бы один положительный таргет
candidates_to_sample = candidates.groupby("user_id").filter(lambda x: x["target"].sum() > 0)

# для каждого пользователя оставляем только 4 негативных примера
negatives_per_user = 4
candidates_for_train = pd.concat([
    candidates_to_sample.query("target == 1"),
    candidates_to_sample.query("target == 0")
        .groupby("user_id")
        .apply(lambda x: x.sample(negatives_per_user, random_state=0))
])

In [40]:
candidates_for_train.shape

(213708, 5)

In [41]:
from catboost import CatBoostClassifier, Pool

# задаём имена колонок признаков и таргета
features = ['als_score', 'cnt_score']
target = 'target'

# Create the Pool object
train_data = Pool(
    data=candidates_for_train[features], 
    label=candidates_for_train[target])

# инициализируем модель CatBoostClassifier
cb_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    loss_function='Logloss',
    verbose=100,
    random_seed=0
)

# тренируем модель
cb_model.fit(train_data)

0:	learn: 0.6526057	total: 72.7ms	remaining: 1m 12s
100:	learn: 0.5118959	total: 1.87s	remaining: 16.7s
200:	learn: 0.5111710	total: 3.76s	remaining: 15s
300:	learn: 0.5105208	total: 5.66s	remaining: 13.2s
400:	learn: 0.5100174	total: 7.55s	remaining: 11.3s
500:	learn: 0.5095747	total: 9.43s	remaining: 9.39s
600:	learn: 0.5091600	total: 11.3s	remaining: 7.51s
700:	learn: 0.5087803	total: 13.2s	remaining: 5.63s
800:	learn: 0.5084220	total: 15.1s	remaining: 3.74s
900:	learn: 0.5080930	total: 16.9s	remaining: 1.86s
999:	learn: 0.5078081	total: 18.8s	remaining: 0us


In [43]:
# загружаем рекомендации от двух базовых генераторов
als_recommendations_2 = pd.read_parquet("goodsread/candidates/inference/als_recommendations.parquet")
content_recommendations_2 = pd.read_parquet("goodsread/candidates/inference/content_recommendations.parquet")

candidates_to_rank = pd.merge(
    als_recommendations_2[["user_id", "item_id", "score"]].rename(columns={"score": "als_score"}),
    content_recommendations_2[["user_id", "item_id", "score"]].rename(columns={"score": "cnt_score"}),
    on=["user_id", "item_id"],
    how="outer"
)

# оставляем только тех пользователей, что есть в тестовой выборке, для экономии ресурсов
candidates_to_rank = candidates_to_rank[candidates_to_rank["user_id"].isin(events_test_2["user_id"].drop_duplicates())]
print(len(candidates_to_rank))

14517152


In [44]:
candidates_to_rank.shape

(14517152, 4)

In [45]:
# Ранжирование кандидатов
inference_data = Pool(data=candidates_to_rank[features])
predictions = cb_model.predict_proba(inference_data)

candidates_to_rank["cb_score"] = predictions[:, 1]

# для каждого пользователя проставляем rank, начиная с 1 — это максимальный cb_score
candidates_to_rank = candidates_to_rank.sort_values(["user_id", "cb_score"], ascending=[True, False])
candidates_to_rank["rank"] = candidates_to_rank.groupby("user_id").cumcount() + 1

max_recommendations_per_user = 100
final_recommendations = candidates_to_rank[candidates_to_rank["rank"] <= max_recommendations_per_user]

In [46]:
final_recommendations.shape

(7519400, 6)

In [47]:
# Валидация 
events_inference = pd.concat([events_train, events_labels])

cb_events_recs_for_binary_metrics_5 = process_events_recs_for_binary_metrics(
    events_inference,
    events_test_2,
    final_recommendations.rename(columns={"cb_score": "score"}),
    top_k=5
)

cb_precision_5, cb_recall_5 = compute_cls_metrics(cb_events_recs_for_binary_metrics_5)

print(f"precision: {cb_precision_5:.3f}, recall: {cb_recall_5:.3f}")

Common users: 75194
precision: 0.006, recall: 0.015


# === Двухстадийный подход: построение признаков

In [48]:
items["age"] = 2018 - items["publication_year"]
invalid_age_idx = items["age"] < 0
items.loc[invalid_age_idx, "age"] = np.nan
items["age"] = items["age"].astype("float")

print(items.columns)

Index(['item_id', 'author', 'title', 'description', 'genre_and_votes',
       'num_pages', 'average_rating', 'ratings_count', 'text_reviews_count',
       'publisher', 'publication_year', 'country_code', 'language_code',
       'format', 'is_ebook', 'isbn', 'isbn13', 'genre_and_votes_dict',
       'genre_and_votes_str', 'item_id_enc', 'age'],
      dtype='object')


In [49]:
candidates_for_train = candidates_for_train.merge(
    items[["item_id", "age"]],
    on="item_id",
    how="left"
)

candidates_to_rank = candidates_to_rank.merge(
    items[["item_id", "age"]],
    on="item_id",
    how="left"
)

In [ ]:
# Медианный возраст
candidates_to_rank["age"].median()

7.0